# F1 2021-2026 EDA and Model Architecture

This notebook explores historical (2021-2025) and 2026 data, highlights changes across regulation eras, and documents the blended LambdaRank modeling plan.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_DIR = Path("data")

In [ ]:
historical_path = DATA_DIR / "historical_races.csv"
results_2026_path = DATA_DIR / "2026_race_results.csv"

historical = pd.read_csv(historical_path) if historical_path.exists() else pd.DataFrame()
results_2026 = pd.read_csv(results_2026_path) if results_2026_path.exists() else pd.DataFrame()

print("Historical rows:", len(historical))
print("2026 rows:", len(results_2026))

## Data Completeness Audit

In [ ]:
if not historical.empty:
    seasons = sorted(historical["season"].dropna().unique().tolist())
    rounds_per_season = historical.groupby("season")["round"].max().to_dict()
    drivers_per_round = historical.groupby(["season", "round"])["driver_id"].nunique()
    incomplete = drivers_per_round[drivers_per_round < 20]
    print("Seasons:", seasons)
    print("Rounds per season:", rounds_per_season)
    if incomplete.empty:
        print("All historical rounds have >= 20 drivers.")
    else:
        print("Rounds with < 20 drivers:")
        display(incomplete.head(20))
else:
    print("Historical data not loaded.")

## Combine Historical + 2026 for EDA

In [ ]:
def normalize_results(df, season_override=None):
    if df.empty:
        return df
    df = df.copy()
    if season_override is not None:
        df["season"] = season_override
    for col in ["season", "round", "grid_position", "finish_position"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "points" in df.columns:
        df["points"] = pd.to_numeric(df["points"], errors="coerce").fillna(0.0)
    return df

historical_norm = normalize_results(historical)
results_2026_norm = normalize_results(results_2026, season_override=2026)
combined = pd.concat([historical_norm, results_2026_norm], ignore_index=True)
combined = combined.dropna(subset=["season", "round", "driver_id"]) if not combined.empty else combined
combined["season"] = combined["season"].astype(int)
combined.head()

## Trend Comparisons: 2021-2025 vs 2026

In [ ]:
if not combined.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.boxplot(data=combined, x="season", y="finish_position", ax=axes[0])
    axes[0].set_title("Finish Position Distribution by Season")
    sns.boxplot(data=combined, x="season", y="points", ax=axes[1])
    axes[1].set_title("Points Distribution by Season")
    plt.tight_layout()
else:
    print("Combined dataset is empty.")

In [ ]:
if "grid_position" in combined.columns and "finish_position" in combined.columns:
    combined["grid_to_finish"] = combined["grid_position"] - combined["finish_position"]
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.boxplot(data=combined, x="season", y="grid_to_finish", ax=ax)
    ax.set_title("Grid to Finish Delta by Season (Positive = Gain Positions)")
    plt.tight_layout()
else:
    print("Missing grid_position or finish_position for grid-to-finish analysis.")

## Driver and Constructor Shifts

In [ ]:
if not combined.empty and "constructor_name" in combined.columns:
    constructor_points = (
        combined.groupby(["season", "constructor_name"])["points"]
        .mean()
        .reset_index()
    )
    latest = constructor_points[constructor_points["season"] == combined["season"].max()]
    top_teams = latest.sort_values("points", ascending=False).head(6)["constructor_name"].tolist()
    focus = constructor_points[constructor_points["constructor_name"].isin(top_teams)]

    plt.figure(figsize=(10, 5))
    sns.lineplot(data=focus, x="season", y="points", hue="constructor_name", marker="o")
    plt.title("Average Points per Race by Constructor (Top Teams)")
    plt.tight_layout()
else:
    print("Constructor analysis not available.")

In [ ]:
if not results_2026_norm.empty and not historical_norm.empty:
    hist_drivers = set(historical_norm["driver_id"].dropna().unique())
    current_drivers = set(results_2026_norm["driver_id"].dropna().unique())
    rookies = sorted(current_drivers - hist_drivers)
    print("Drivers in 2026 not in historical data:", rookies)
else:
    print("Rookie analysis not available.")

## Regulation-Era Comparison (2021-2022 vs 2023-2025 vs 2026)

In [ ]:
def assign_era(season):
    if season <= 2022:
        return "2021-2022"
    if season <= 2025:
        return "2023-2025"
    return "2026"

if not combined.empty:
    combined["era"] = combined["season"].apply(assign_era)
    era_summary = combined.groupby("era")["points"].agg(["mean", "std"]).reset_index()
    display(era_summary)
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=combined, x="era", y="points")
    plt.title("Points Distribution by Regulation Era")
    plt.tight_layout()
else:
    print("Era comparison not available.")

## Narrative Summary (Why We Need a New Model)

- 2026 has limited rounds and shows higher variance in early-season form, so a pure 2026 model is unstable.
- Historical performance patterns still matter, but regulation changes (2022 and 2026) shift competitive balance.
- Driver/constructor churn (rookies and team switches) means older seasons can mislead unless we decay or blend.
- A blended LambdaRank approach gives stability from history while staying responsive to 2026 form.

## Model Architecture Plan (Detailed)

**Pipeline**
1. Ingestion: FastF1 historical results (2021-2025) with Ergast fallback; 2026 results + standings.
2. Canonical normalization: enforce columns (season, round, driver_id, constructor_id, grid_position, finish_position, points, race_name, circuit_id, source).
3. Feature engineering: pre-qualifying features using only past rounds within each season to avoid leakage.
4. Historical LambdaRank model: train on 2021-2025 with recency decay and regulation dampeners.
5. 2026 LambdaRank model: train on 2026 rounds before the target race.
6. Blending: combine scores with alpha (default 0.65 for 2026, 0.35 for historical).
7. Output: ranked predictions + component scores for traceability.

**Feature columns**
- current_points, current_position, wins, podiums, poles
- avg_finish_position, avg_points_per_race, races_completed
- last_race_finish, last_race_points, momentum
- constructor_avg_finish, constructor_avg_points

**Weighting strategy**
- Season decay: 2025=1.0, 2024=0.7, 2023=0.5, 2022=0.3, 2021=0.2
- Regulation dampener: multiply pre-2022 weights by 0.7
- Blend alpha: 0.65 (2026) / 0.35 (historical), tune with backtests

**Evaluation**
- NDCG for ranking quality
- Top-3 hit rate for podium accuracy
- Backtest on 2026 rounds (train on earlier rounds, predict later rounds)